In [8]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit
import os
import re
import dotenv
import logging
from logging.handlers import RotatingFileHandler
from pathlib import Path
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from delta import configure_spark_with_delta_pip
import datetime
import pandas as pd


from ingest_helpers.file_info_helpers import *
from ingest_helpers.spark_df_helpers import *
from ingest_helpers.config_helpers import *


## config loading

dotenv.load_dotenv()




## Path setup



LOG_DIR = Path(os.getenv("LOG_DIR", "/logs"))
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = LOG_DIR / "discogs_bronze_ingest.log"

LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE.touch(exist_ok=True)

DATA_DIR = Path(os.getenv("DATA_DIR", "/data_tests"))

RAW_DATA_DIR = DATA_DIR / "raw"


BRONZE_DATA_DIR = DATA_DIR / "bronze"
BRONZE_DATA_DIR.mkdir(parents=True, exist_ok=True)

archive_dir = RAW_DATA_DIR / "archive"
archive_dir.mkdir(parents=True, exist_ok=True)



# Loading bronze config


CONFIG_DIR = Path(os.getenv("CONFIG_DIR", "/config"))
CONFIG_FILE = CONFIG_DIR / "bronze_config.json"

BRONZE_CONFIG = load_config_from_json(CONFIG_FILE)



In [9]:
BRONZE_CONFIG

{'PRIMARY_KEYS': {'artists': 'id', 'releases': '@id'}}

In [12]:
primary_key = BRONZE_CONFIG['PRIMARY_KEYS']
primary_key



{'artists': 'id', 'releases': '@id'}

In [15]:
EXPORT_HISTORY_TO_CSV = Path(os.getenv("EXPORT_HISTORY_TO_CSV"))
EXPORT_HISTORY_TO_CSV

PosixPath('True')

In [16]:


def setup_logger()-> logging.Logger:


    # Remove any existing handlers first
    root_logger = logging.getLogger()
    for h in root_logger.handlers[:]:
        root_logger.removeHandler(h)

    # File handler
    file_handler = RotatingFileHandler(
        LOG_FILE,
        maxBytes=50 * 1024 * 1024,  # 50 MB
        backupCount=5,
    )
    file_handler.setFormatter(
        logging.Formatter(
            "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
            "%Y-%m-%d %H:%M:%S",
        )
    )

    # Console handler
    console_handler = logging.StreamHandler()
    console_handler.setFormatter(
        logging.Formatter(
            "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
            "%Y-%m-%d %H:%M:%S",
        )
    )

    # Add handlers
    root_logger.setLevel(logging.INFO)
    root_logger.addHandler(file_handler)
    root_logger.addHandler(console_handler)

    logger = logging.getLogger(__name__)
    ## Env and config variables
    return logger






In [17]:


def create_spark_session(app_name: str) -> SparkSession:




    ## Spark session

    # Build the Spark session
    builder = (
        SparkSession.builder.appName("discogs-bronze-ingest")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config(
            "spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog",
        )
    )

    # This injects the Delta jars from delta-spark Python package
    spark = configure_spark_with_delta_pip(builder).getOrCreate()
    return spark






In [19]:
logger = setup_logger()

In [26]:
## Listing files to process
def get_latest_dump_files(raw_dir: Path) -> list:    
    dumps_to_process = []    

    logger.info(f"{RAW_DATA_DIR = }")
    logger.info(f"{BRONZE_DATA_DIR = }")
    logger.info(f"{LOG_DIR = }")
    logger.info(f"{LOG_FILE = }")

    raw_data_file_list = os.listdir(RAW_DATA_DIR)
    logger.info(f"{raw_data_file_list = }")

    dump_types = [extract_dump_type(f) for f in raw_data_file_list]
    dump_types = set([d for d in dump_types if d])


    if not dump_types:
        logger.error(f"No Discogs dump files found in RAW_DATA_DIR")
        raise RuntimeError("No Discogs dump files found in RAW_DATA_DIR")

    for dump_type in dump_types:
        dump_dates = [extract_dump_date(f) for f in raw_data_file_list if dump_type in f]
        dump_dates = [d for d in dump_dates if d]

        if not dump_dates:
            logger.error(f"No Discogs dump files found in RAW_DATA_DIR")
            raise RuntimeError("No Discogs dump files found in RAW_DATA_DIR")

        dump_dates = list(set(dump_dates))
        dump_dates.sort(reverse=True)

        logger.info(f"{dump_type = }")
        logger.info(f"{dump_dates = }")

        latest_dump_date = dump_dates[0]
        logger.info(f"{latest_dump_date = }")

        raw_data_file_list_dump = [
            file_name
            for file_name in raw_data_file_list
            if (latest_dump_date in file_name)
            and (file_name.endswith(".xml.gz") and (dump_type in file_name))
        ]

        if not raw_data_file_list_dump:
            logger.warning(f"No XML files found for dump {latest_dump_date}")

        logger.info(f"{raw_data_file_list_dump = }")

        for file in raw_data_file_list_dump:
            input_file_path = os.path.join(RAW_DATA_DIR, file)
            output_dir = os.path.join(BRONZE_DATA_DIR, dump_type)

            logger.info(f"{input_file_path = }")
            logger.info(f"{output_dir = }")

            output_dir_path = Path(output_dir)
            output_dir_path.mkdir(parents=True, exist_ok=True)


        dumps_to_process.append({ "dump_type":dump_type,  "latest_dump_date":latest_dump_date ,"input_file_path":input_file_path})   
    return dumps_to_process
        

get_latest_dump_files(RAW_DATA_DIR)


2026-02-22 14:59:29 | INFO     | __main__ | RAW_DATA_DIR = PosixPath('/data_tests/raw')
2026-02-22 14:59:29 | INFO     | __main__ | BRONZE_DATA_DIR = PosixPath('/data_tests/bronze')
2026-02-22 14:59:29 | INFO     | __main__ | LOG_DIR = PosixPath('/logs')
2026-02-22 14:59:29 | INFO     | __main__ | LOG_FILE = PosixPath('/logs/discogs_bronze_ingest.log')
2026-02-22 14:59:29 | INFO     | __main__ | raw_data_file_list = ['discogs_20251101_artists_sample_100000.xml.gz', 'discogs_20251201_releases_sample_200000.xml.gz', 'discogs_20251101_releases_sample_100000.xml.gz', 'archive', 'discogs_20251201_artists_sample_200000.xml.gz']
2026-02-22 14:59:29 | INFO     | __main__ | dump_type = 'artists'
2026-02-22 14:59:29 | INFO     | __main__ | dump_dates = ['20251201', '20251101']
2026-02-22 14:59:29 | INFO     | __main__ | latest_dump_date = '20251201'
2026-02-22 14:59:29 | INFO     | __main__ | raw_data_file_list_dump = ['discogs_20251201_artists_sample_200000.xml.gz']
2026-02-22 14:59:29 | INFO  

[{'dump_type': 'artists',
  'latest_dump_date': '20251201',
  'input_file_path': '/data_tests/raw/discogs_20251201_artists_sample_200000.xml.gz'},
 {'dump_type': 'releases',
  'latest_dump_date': '20251201',
  'input_file_path': '/data_tests/raw/discogs_20251201_releases_sample_200000.xml.gz'}]

In [71]:
def extract_dump_type(fname):
    m = re.search(r"discogs_\d{8}_(.*?)[\._]", fname)
    return m.group(1) if m else None

In [76]:
dump_types = [extract_dump_type(f) for f in raw_data_file_list ]
dump_types = set([d for d in dump_types if d])

In [77]:
dump_types

{'artists', 'releases'}

In [83]:
for dumb_type in dump_types:

    dump_dates = [extract_dump_date(f) for f in raw_data_file_list if dumb_type in f]
    dump_dates = [d for d in dump_dates if d]

    if not dump_dates:
        logger.error(f"No Discogs dump files found in RAW_DATA_DIR")
        raise RuntimeError("No Discogs dump files found in RAW_DATA_DIR")


    dump_dates  = list(set(dump_dates))
    dump_dates.sort(reverse=True)


    logger.info(f"{dumb_type = }")
    logger.info(f"{dump_dates = }")


    latest_dump_date = dump_dates[0]
    logger.info(f"{latest_dump_date = }")



    raw_data_file_list_dump = [file_name for file_name in raw_data_file_list if (latest_dump_date in file_name) and (file_name.endswith(".xml.gz")and (dumb_type in file_name))]

    if not raw_data_file_list_dump:
        logger.warning(f"No XML files found for dump {latest_dump_date}")


    logger.info(f"{raw_data_file_list_dump = }")




2026-02-21 15:07:22 | INFO     | __main__ | dumb_type = 'artists'
2026-02-21 15:07:22 | INFO     | __main__ | dump_dates = ['20260101', '20251201', '20251101']
2026-02-21 15:07:22 | INFO     | __main__ | latest_dump_date = '20260101'
2026-02-21 15:07:22 | INFO     | __main__ | raw_data_file_list_dump = ['discogs_20260101_artists_sample_300000.xml.gz']
2026-02-21 15:07:22 | INFO     | __main__ | dumb_type = 'releases'
2026-02-21 15:07:22 | INFO     | __main__ | dump_dates = ['20260101', '20251201', '20251101']
2026-02-21 15:07:22 | INFO     | __main__ | latest_dump_date = '20260101'
2026-02-21 15:07:22 | INFO     | __main__ | raw_data_file_list_dump = ['discogs_20260101_releases_sample_300000.xml.gz']
